# 09. Мини-исследование π-ID: frozen teacher → обучаемая DX policy

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 2](https://diffusion.csail.mit.edu/2026/docs/20260122_Lecture_02.pdf).
<!-- study-navigation:end -->

Проведём маленький воспроизводимый training experiment с известным 2D teacher field, held-out probe и stop-gradient rollout. Промышленные weights не нужны.

[Маршрут](../README.md) · [Точные задания MIT](../MIT_BRIDGE.md) · [Методы и source map](../SOURCE_MAP.md) · [Измерения](../reports/VALIDATION.md).

Каждый опыт: **теория → формула → формы/единицы → прогноз → код → измерение → ограниченный вывод**. CPU по умолчанию; без pretrained weights, сети и ComfyUI jobs.


**Связь с MIT:** MIT Lab 2 Q3.1, notes §3.3 pp.19–24: velocity regression. В π-ID target — frozen teacher на rollout student; это другой training distribution и отдельная статья.

[MIT course](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf).


In [ ]:
from pathlib import Path
import sys,math,json,copy
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
COURSE=next((p/'dit' for p in [Path.cwd(),*Path.cwd().parents] if (p/'dit/labkit/core.py').is_file()),None)
assert COURSE is not None,'Clone the study repository first; see dit/README.md (Colab setup).'
sys.path.insert(0,str(COURSE))
from labkit.core import (native,tiny_model,tiny_inputs,activate_gates,fresh_forward,
  load_k5_block,manual_single,manual_fused,CountingLinearField,tiny_bundle,
  clone_bundle,scalar_bundle,vector_from_bundle,dummy_text,show_source,
  check_sources,save_metrics,figure_path)
torch.set_num_threads(2);torch.manual_seed(42)
LAB='09_miniature_policy_imitation_distillation';metrics={}
print('Pinned source files checked:',check_sources())


## 1. FM regression и imitation — разные targets

MIT CFM regresses доступный conditional target ε−z на data/noise paths. Здесь teacher Gθ уже задаёт поле; student Gφ выдаёт policy parameters η. На начале segment строим η=Gφ(x_s,σ_s), а промежуточные states получаем по **detached student policy**:

$$
\tilde x_\sigma=\mathrm{Rollout}(\mathrm{stopgrad}(\eta),x_s,\sigma_s\to\sigma).
$$

Затем сравниваем live policy velocity с frozen teacher:

$$
L(\phi)=\mathbb{E}\left[\|\pi_\eta(\mathrm{stopgrad}(\tilde x_\sigma),\sigma)
-\mathrm{stopgrad}(G_\theta(\tilde x_\sigma,\sigma))\|^2\right].
$$

Gradient идёт к live η, но не через teacher или state-generating rollout. Это позволяет обучать поведение на student-induced distribution. Fixed off-policy states и on-policy states отвечают на разные вопросы; ниже training on-policy, evaluation — дополнительно fixed held-out teacher states, чтобы изменение distribution не маскировало loss.

Первичный источник: [π-Flow §4 / Algorithm 1](https://arxiv.org/html/2510.14974v1#S4). Наш optimizer/dataset/MLP — учебная постановка, не reproduction K6 recipe и не release training script. В paper есть также micro-window loss и GMFlow; здесь изучаем базовый instantaneous DX objective.


In [ ]:
pm=native('core.algo.piflow_math');torch.manual_seed(23)
A=torch.tensor([[.7,.2],[.3,.4]])
teacher=nn.Linear(2,2,bias=False);teacher.weight.data.copy_(A);teacher.requires_grad_(False)
G=5
student=nn.Sequential(nn.Linear(3,64),nn.SiLU(),nn.Linear(64,G*2))
opt=torch.optim.Adam(student.parameters(),lr=.003)
def parameters_for(xs,ss):return student(torch.cat((xs,ss[:,None]),-1)).reshape(-1,G,2)
def draw(batch,generator=None):
    xs=torch.randn(batch,2,generator=generator)
    ss=torch.tensor([.25,.5,.75,1.])[torch.randint(0,4,(batch,),generator=generator)]
    query=ss-.25*(.05+.9*torch.rand(batch,generator=generator))
    return xs,ss,query
# Held-out values fixed before training; independent generator.
xs_h,ss_h,tq_h=draw(512,torch.Generator().manual_seed(1001))
xq_h=(torch.matrix_exp((tq_h-ss_h)[:,None,None]*A)@xs_h[:,:,None]).squeeze(-1)
with torch.no_grad():target_h=teacher(xq_h)
def fixed_probe():
    with torch.no_grad():
        p=pm.DXPolicy(parameters_for(xs_h,ss_h),xs_h,ss_h,segment_size=.25)
        return float(F.mse_loss(p.pi(xq_h,tq_h),target_h))
before=fixed_probe();curve=[];teacher_before=teacher.weight.detach().clone()


## 2. Train → measure, без утечки held-out target

Каждый training step получает новые source states и времена. Исходный `policy_rollout_fm` интегрирует detached policy до query time. Teacher evaluated там, где оказался student; training query states не берутся из аналитической teacher trajectory. Зафиксированный held-out probe используется только для оценки.

Проверяем routing градиентов: teacher.grad=None, rollout state requires_grad=False, live policy coefficients передают gradient student weights. Уменьшение loss свидетельствует об imitation на этом toy field, а не о generation quality K6.


In [ ]:
for step in range(500):
    xs,ss,tq=draw(64)
    p=pm.DXPolicy(parameters_for(xs,ss),xs,ss,segment_size=.25)
    with torch.no_grad():
        xq,sigma,_=pm.policy_rollout_fm(xs,ss,ss,tq,32,p.detach())
        target=teacher(xq)
    pred=p.pi(xq,sigma.flatten())
    loss=F.mse_loss(pred,target)
    opt.zero_grad();loss.backward();torch.nn.utils.clip_grad_norm_(student.parameters(),10.);opt.step()
    curve.append(float(loss.detach()))
after=fixed_probe()
assert not xq.requires_grad and teacher.weight.grad is None and torch.equal(teacher.weight,teacher_before)
assert after<before*.8 and np.isfinite(curve).all()
metrics['toy_pi_ID']={'steps':500,'grid_points':G,'heldout_before':before,'heldout_after':after,
 'improvement_factor':before/max(after,1e-30),'first_50_train_mean':float(np.mean(curve[:50])),
 'last_50_train_mean':float(np.mean(curve[-50:])),'teacher_frozen':True,'rollout_detached':True,
 'teacher_training_batch_calls':500,'teacher_training_states':500*64,'student_source_training_batch_calls':500}
print(metrics['toy_pi_ID'])
fig,axes=plt.subplots(1,2,figsize=(10,3));axes[0].plot(curve,lw=.8);axes[0].set(xlabel='Optimizer step',ylabel='On-policy velocity MSE')
axes[1].bar(['Before','After'],[before,after]);axes[1].set(yscale='log',ylabel='Fixed held-out velocity MSE');fig.tight_layout();fig.savefig(figure_path(LAB,'imitation_training'));plt.show()


## 3. От velocity loss к trajectory error: что требует доказательства

Если teacher Lipschitz по x с константой L и policy error ≤ε на всём relevant rollout domain, Grönwall даёт на interval длины Δ:

$$
\|x_{student}(\sigma)-x_{teacher}(\sigma)\|
\leq e^{L\Delta}\|\delta x_s\|+\frac{e^{L\Delta}-1}{L}\varepsilon
+\mathrm{integration\ error}.
$$

Это bound с предпосылками, а не автоматическое следствие малого **среднего** train loss. Надо проверять held-out trajectories и near-endpoint error отдельно. У DX около σ=0 Lipschitz/anchor sensitivity может расти; eps задаёт numerical clamp, но не доказывает fidelity.

Измерим trajectory states на held-out sources до нескольких positive query times, используя source interval .25. Teacher exact matrix exponential. В отличие от только instantaneous regression, это проверяет накопленную ошибку rollout.


In [ ]:
rows=[]
with torch.no_grad():
    xs=xs_h[:64];ss=ss_h[:64]
    p=pm.DXPolicy(parameters_for(xs,ss),xs,ss,segment_size=.25)
    for fraction in [.25,.5,.9]:
        tq=ss-.25*fraction
        out,_,_=pm.policy_rollout_fm(xs,ss,ss,tq,128,p.detach())
        exact=(torch.matrix_exp((tq-ss)[:,None,None]*A)@xs[:,:,None]).squeeze(-1)
        rows.append({'fraction_of_segment':fraction,'state_MSE':float(F.mse_loss(out,exact)),
                     'network_calls_per_source_segment':1,'policy_density':128})
metrics['heldout_trajectory_probe']=rows
for row in rows:print(row)


## 4. Что в этом эксперименте не воспроизведено

K6 §4.5 описывает initialization из pretrained backbone, отдельные training regimes и adversarial post-training. Здесь teacher — известная матрица, student — маленький MLP, нет text/video/audio данных, real teacher CFG, perceptual discriminator или Sim-LADD. Мы проверили механизм gradient/rollout/imitation и получили численные результаты с явным scope.

Для применения к CrossDiT понадобятся paired modalities, реальные pretrained heads/parameter layouts, teacher-query accounting, memory budget и held-out perceptual/regression metrics. Меньше NFE без соответствующего обучения не является distillation.


## Измерение → вывод

Frozen teacher сохранил веса, rollout был detached, а live policy обучалась. Fixed held-out velocity loss уменьшился; trajectory error измерена отдельно. Это маленький numerical imitation experiment, не Kandinsky distillation training и не proof of pretrained quality.


## Самостоятельные опыты

1. Повторите с 3 seeds и сравните fixed held-out loss, не только train curve.
2. Измените G при том же optimizer budget.
3. Сравните off-policy и on-policy state distributions на одной teacher matrix.
4. Уберите detach только в отдельном опыте и измерьте memory/gradient changes; не называйте это тем же Alg.1.
5. Почему held-out mean loss недостаточен для uniform Grönwall bound?

Перед изменением запишите прогноз. Сохраняйте baseline, меняйте один фактор, отмечайте seed и tensor layout. Нулевая ошибка на oracle не равна pretrained quality.


In [ ]:
result=save_metrics(LAB,metrics)
print(json.dumps(result,ensure_ascii=False,indent=2))


## Первичные источники

[MIT course / notes](https://diffusion.csail.mit.edu/2026/index.html) · [DiT](https://arxiv.org/abs/2212.09748) · [Flow Matching](https://arxiv.org/abs/2210.02747) · [K5](https://arxiv.org/abs/2511.14993) · [K6 §3.2, §4.5](https://arxiv.org/html/2610.05608v1) · [π-Flow §3.1, §4 / Alg.1](https://arxiv.org/html/2510.14974v1) · [Source hashes / provenance](../sources.json).

Формулы ниже/выше выводятся для явно заданных соглашений. Тела source methods сохраняются; adapters и doubles подписаны. Малые experiments не воспроизводят промышленное обучение.
